In [84]:
# pdf upload
pdf_path = '../data/Medical_book.pdf'

In [85]:
# text extraction
from pypdf import PdfReader

reader = PdfReader(pdf_path)

full_text = ""

for page in reader.pages:
    text = page.extract_text()

    if text:
        full_text += text + "\n"

print(full_text)

The GALE
ENCYCLOPEDIA
of MEDICINE
SECOND EDITION
The GALE
ENCYCLOPEDIA
of MEDICINE
SECOND EDITION
JACQUELINE L. LONGE, EDITOR
DEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR
VOLUME
A-B
1
STAFF
Jacqueline L. Longe, Project Editor
Deirdre S. Blanchfield, Associate Editor
Christine B. Jeryan, Managing Editor
Donna Olendorf, Senior Editor
Stacey Blachford, Associate Editor
Kate Kretschmann, Melissa C. McDade, Ryan
Thomason, Assistant Editors
Mark Springer, Technical Specialist
Andrea Lopeman, Programmer/Analyst
Barbara J. Yarrow,Manager, Imaging and Multimedia
Content
Robyn V . Young,Project Manager, Imaging and
Multimedia Content
Dean Dauphinais, Senior Editor, Imaging and
Multimedia Content
Kelly A. Quin, Editor, Imaging and Multimedia Content
Leitha Etheridge-Sims, Mary K. Grimes, Dave Oblender,
Image Catalogers
Pamela A. Reed, Imaging Coordinator
Randy Bassett, Imaging Supervisor
Robert Duncan, Senior Imaging Specialist
Dan Newell, Imaging Specialist
Christine O’Bryan, Graphic Specialist
Mari

In [86]:
print(len(full_text))

2630696


In [87]:
# chunking
def chunk_text(text, chunk_size=500, chunk_overlap=50):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end]
        chunks.append(chunk)

        start += chunk_size - chunk_overlap

    return chunks

chunks = chunk_text(full_text,
                    chunk_size=500,
                    chunk_overlap=50)

print(chunks[440])

 current through some of them. Moxibustion may
be sometimes used, in which an herbal mixture (moxa or
mugwort) is either burned like incense on the acupuncture
point or on the end of the needle, which is believed to stim-
ulate chi in a particular way. Also, acupuncturists some-
times use cupping, during which small suction cups are
placed on meridian points to stimulate them.
How long the needles are inserted also varies. Some
patients only require a quick in and out insertion to clear
problems


In [88]:
# embedding
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(chunks)

print(f"Number of embeddings: {len(embeddings)}")

print("\nEmbedding dimension:")
print(len(embeddings[0]))

print("\nFirst embedding:")
print(embeddings[0])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6757.70it/s]


Number of embeddings: 5846

Embedding dimension:
384

First embedding:
[ 4.58733775e-02  3.50594381e-03 -1.91168897e-02  6.40005991e-03
  1.60799716e-02  6.18166476e-02 -7.41795301e-02  1.78339079e-01
  3.10915057e-02 -2.81434748e-02 -1.54379467e-02  8.14737901e-02
  3.29385437e-02  2.05649063e-02 -5.72270192e-02  1.63698550e-02
 -3.12609002e-02 -7.15976506e-02 -5.72776422e-02 -4.17282106e-03
 -5.90430759e-02  5.86228333e-02  3.35893035e-02  6.57212734e-02
 -5.31884693e-02  2.83759069e-02 -5.55800609e-02 -4.92508411e-02
 -5.83993085e-02  1.89645253e-02  3.30646597e-02  1.11685000e-01
  7.80687779e-02  1.26149133e-02 -3.26022925e-03 -1.36528397e-02
 -2.63424050e-02  4.69710045e-02 -7.97686130e-02  6.99774325e-02
  6.71024853e-03 -7.61430934e-02 -6.99955970e-02  1.41259553e-02
  6.81322515e-02 -4.63351272e-02 -2.86262389e-02 -8.02208669e-03
  3.10271829e-02  9.62517038e-02 -1.20146215e-01 -6.47143200e-02
 -3.31070125e-02  7.67451078e-02  1.94637105e-02  3.94264273e-02
 -5.62285893e-02 -5

In [89]:
# opensearch indexing
from opensearchpy import OpenSearch

client = OpenSearch(
    hosts=[{
        "host": "localhost",
        "port": 9200
    }],
    use_ssl=False,
    verify_certs=False
)

client.info()

{'name': 'c29840328ae3',
 'cluster_name': 'docker-cluster',
 'cluster_uuid': 'XfYMnE0RQUiiGH5qt9VyAg',
 'version': {'distribution': 'opensearch',
  'number': '3.6.0',
  'build_type': 'tar',
  'build_hash': '4ca747d8d47f80162db323019357447126732e35',
  'build_date': '2026-04-04T06:02:53.715432718Z',
  'build_snapshot': False,
  'lucene_version': '10.4.0',
  'minimum_wire_compatibility_version': '2.19.0',
  'minimum_index_compatibility_version': '2.0.0'},
 'tagline': 'The OpenSearch Project: https://opensearch.org/'}

In [90]:
# create opensearch index
INDEX_NAME = "document-index"

index_body = {
    "settings": {
        "index": {
            "knn": True
        }
    },
    "mappings": {
        "properties": {
            "text": {
                "type": "text"
            },
            "embedding": {
                "type": "knn_vector",
                "dimension": 384
            }
        }
    }
}

if not client.indices.exists(index=INDEX_NAME):
    client.indices.create(
        index=INDEX_NAME,
        body=index_body
    )

    print("Index created")
else:
    print("Index already exists")

Index already exists


In [91]:
documents = []

for chunk, embedding in zip(chunks, embeddings):

    documents.append({
        "text": chunk,
        "embedding": embedding.tolist()
    })

for i, doc in enumerate(documents):

    client.index(
        index=INDEX_NAME,
        id=i,
        body=doc
    )

print("Documents indexed successfully")

Documents indexed successfully


In [92]:
client.indices.refresh(index=INDEX_NAME)

{'_shards': {'total': 2, 'successful': 1, 'failed': 0}}

In [93]:
response = client.search(
    index=INDEX_NAME,
    body={
        "query": {
            "match_all": {}
        }
    }
)

print(response)

{'took': 2, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 5846, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'document-index', '_id': '0', '_score': 1.0, '_source': {'text': 'The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION\nThe GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\nA-B\n1\nSTAFF\nJacqueline L. Longe, Project Editor\nDeirdre S. Blanchfield, Associate Editor\nChristine B. Jeryan, Managing Editor\nDonna Olendorf, Senior Editor\nStacey Blachford, Associate Editor\nKate Kretschmann, Melissa C. McDade, Ryan\nThomason, Assistant Editors\nMark Springer, Technical Specialist\nAndrea Lopeman, Programmer/An', 'embedding': [0.045873377, 0.0035059438, -0.01911689, 0.00640006, 0.016079972, 0.061816648, -0.07417953, 0.17833908, 0.031091506, -0.028143475, -0.015437947, 0.08147379, 0.032938544, 0.020564906, -0.05722702, 0.016369

In [94]:
# query and retrieval
query = "What is acne?"

query_embedding = model.encode(query)

print(query_embedding.shape)

(384,)


In [95]:
response = client.search(
    index=INDEX_NAME,
    body={
        "size": 5,
        "query": {
            "knn": {
                "embedding": {
                    "vector": query_embedding.tolist(),
                    "k": 5
                }
            }
        }
    }
)

In [96]:
for i, hit in enumerate(response["hits"]["hits"]):

    print(f"\n--- Retrieved Chunk {i+1} ---\n")

    print(hit["_source"]["text"])

    print("\nScore:")
    print(hit["_score"])


--- Retrieved Chunk 1 ---

zoyl peroxide or tretinoin may be
used to clear up mild to moderately severe acne.
Isotretinoin (Accutane) is prescribed only for very
severe, disfiguring acne.
Acne is a skin condition that occurs when pores or
hair follicles become blocked. This allows a waxy
material, sebum, to collect inside the pores or follicles.
Normally, sebum flows out onto the skin and hair to
form a protective coating, but when it cannot get out,
small swellings develop on the skin surface. Bacteria
and dead skin cell

Score:
0.59743226

--- Retrieved Chunk 2 ---

alch, James F., and Phyllis A. Balch. “The Disorders: Acne.”
In Prescription for Nutritional Healing, ed. Amy C. Teck-
lenburg, et al. New York: Avery Publishing Group, 1997.
Bark, Joseph P. Your Skin: An Owner’s Guide.Englewood
Cliffs, NJ: Prentice Hall, 1995.
Goldstein, Sanford M., and Richard B. Odom. “Skin &
Appendages: Pustular Disorders.” In Current Medical
Diagnosis and Treatment, 1996.35th ed. Ed. Stephen
McPhee,

In [97]:
# llm generation
retrieved_chunks = []

for hit in response["hits"]["hits"]:
    retrieved_chunks.append(hit["_source"]["text"])

context = "\n\n".join(retrieved_chunks)

print(context[:1000])

zoyl peroxide or tretinoin may be
used to clear up mild to moderately severe acne.
Isotretinoin (Accutane) is prescribed only for very
severe, disfiguring acne.
Acne is a skin condition that occurs when pores or
hair follicles become blocked. This allows a waxy
material, sebum, to collect inside the pores or follicles.
Normally, sebum flows out onto the skin and hair to
form a protective coating, but when it cannot get out,
small swellings develop on the skin surface. Bacteria
and dead skin cell

alch, James F., and Phyllis A. Balch. “The Disorders: Acne.”
In Prescription for Nutritional Healing, ed. Amy C. Teck-
lenburg, et al. New York: Avery Publishing Group, 1997.
Bark, Joseph P. Your Skin: An Owner’s Guide.Englewood
Cliffs, NJ: Prentice Hall, 1995.
Goldstein, Sanford M., and Richard B. Odom. “Skin &
Appendages: Pustular Disorders.” In Current Medical
Diagnosis and Treatment, 1996.35th ed. Ed. Stephen
McPhee, et al. Stamford: Appleton & Lange, 1995.
Kaptchuk, Ted J., Z’ev Rosenberg

In [98]:
count = client.count(index=INDEX_NAME)

print(count)

{'count': 5846, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}}


In [99]:
# prompt
query = query

prompt = f"""
You are an enterprise document assistant.

Answer the user's question ONLY using the provided context.

If the answer is not present in the context, say:
"I could not find the relevant information."

Context:
{context}

Question:
{query}

Answer:
"""

In [100]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [101]:
from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)

In [102]:
response = openai_client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

answer = response.choices[0].message.content

print(answer)

Acne is a skin condition that occurs when pores or hair follicles become blocked, allowing a waxy material called sebum to collect inside. Normally, sebum flows out onto the skin and hair to form a protective coating, but when it cannot get out, small swellings develop on the skin surface. Acne involves inflammation of the sebaceous glands and can include mild noninflammatory types like whiteheads and blackheads, as well as moderate to severe inflammatory types caused by bacteria invading the plugged follicle.
